In [1]:
import re
from tqdm import tqdm #Progress Indication
from pprint import pprint,pformat #Saubere JSON darstellung

import numpy as np 
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from collections import defaultdict
from collections import Counter

import spacy # Lemmatizer

import pandas as pd
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.decomposition import TruncatedSVD, LatentDirichletAllocation

from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from GerVADER.vaderSentimentGER import SentimentIntensityAnalyzer as GerSentimentIntensityAnalyzer

import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

**Beschriebenes Vorgehen:**

Bereinigung durch:
- Pandas
- NLTK
- SpaCy (besser für Deutsches Lemmatizing)

-> Separat für Topic Modeling/ Semantik Analyse


Vektorisierung anhand BoW und TF-IDF + n-Gramme durch:
- sklearn

-> Darstellung Unterschied zwischen BoW u TF-IDF
-> TF-IDF sprachsepariert, da sonst falsche stoppwörter


Themenidentifikation (LSA/LDA) durch:
- sklearn
- vaderSentiment
- GerVADER

-> Ausarbeitung bester Ansatz:

BoW + LDA
TF-IDF + LDA
-> TF-IDF + LSA



In [2]:
def main():

    # To be filled   
    print("To be filled")

if __name__ == "__main__":
    main()

To be filled


In [64]:
### Style Funktionen #######

def base_table_style(df, caption=None, extra_styles=None):
    styles = [
        {
            "selector": "table",
            "props": [
                ("table-layout", "fixed"),
                ("width", "100%")
            ]
        },
        {
            "selector": "th",
            "props": [
                ("text-align", "center"),
                ("font-weight", "bold")
            ]
        },
        {
            "selector": "td",
            "props": [
                ("white-space", "normal"),
                ("overflow-wrap", "break-word"),
                ("word-wrap", "break-word"),
                ("padding", "4px"),
                ("border-right", "1px solid lightgray")
            ]
        },
        {
            "selector": "caption",
            "props": [
                ("caption-side", "top"),
                ("text-align", "center"),
                ("font-weight", "bold"),
                ("font-size", "16px")
            ]
        }
    ]

    if extra_styles:
        styles.extend(extra_styles)

    styled = (
        df.style
        .hide(axis="index")
        .set_table_styles(styles)
        .set_properties(**{"text-align": "left"})
    )

    if caption:
        styled = styled.set_caption(caption)

    return styled

def style_topic_comparison(df, caption=None):
    method_start_positions = [
        pos for pos, value in enumerate(df["Method"])
        if value != ""
    ]

    def add_separator(row):
        pos = df.index.get_loc(row.name)

        if pos in method_start_positions and pos != 0:
            return ["border-top: 3px solid black"] * len(row)

        return [""] * len(row)

    return (
        base_table_style(
            df,
            caption=caption
        )
        .apply(add_separator, axis=1)
    )
    
def style_compare_top_tokens(df, caption=None):
    count_cols = [
        ("de", "Topic", "Count"),
        ("de", "Sentiment", "Count"),
        ("en", "Topic", "Count"),
        ("en", "Sentiment", "Count"),
    ]

    extra_styles = [
        {"selector": "th.col1, td.col1", "props": [("border-right", "1px solid black")]},
        {"selector": "th.col3, td.col3", "props": [("border-right", "4px solid black")]},
        {"selector": "th.col5, td.col5", "props": [("border-right", "1px solid black")]},
    ]

    return (
        base_table_style(df, caption, extra_styles)
        .set_properties(subset=count_cols, **{"text-align": "right"})
        .background_gradient(subset=count_cols, cmap="Blues")
    )

def style_topic_sentiment(
    df,
    caption,
    pos_threshold=0.3,
    neg_threshold=-0.3,
    pos_color="#c6efce",
    neg_color="#ffc7ce"
):
    """
    Styled Topic/Sentiment Tabelle mit:
    - Caption
    - konfigurierbaren Schwellenwerten
    - konfigurierbaren Farben
    """

    #caption = f"{method} ({lang}) – Topic-Verteilung & Sentiment"

    # --- Sentiment Highlight ---
    def highlight_sentiment(val):
        if val >= pos_threshold:
            return f"background-color: {pos_color}"
        elif val <= neg_threshold:
            return f"background-color: {neg_color}"
        return ""

    # --- Dominantes Sentiment hervorheben ---
    def highlight_dominance(row):
        max_val = max(row["positive"], row["neutral"], row["negative"])
        return [
            "font-weight: bold" if v == max_val else ""
            for v in [row["positive"], row["neutral"], row["negative"]]
        ]

    styled = base_table_style(df, caption=caption)

    styled = styled.map(highlight_sentiment, subset=["avg_sentiment"])

    styled = styled.apply(
        highlight_dominance,
        axis=1,
        subset=["positive", "neutral", "negative"]
    )
        # Kein Umbruch für Keywords
    if "topic_keywords" in df.columns:
        styled = styled.set_properties(
            subset=["topic_keywords"],
            **{
                "white-space": "nowrap",
                "word-wrap": "normal",
                "overflow-wrap": "normal",
                "min-width": "800px"
            }
        )

    return styled

In [18]:
#### Analyse Funktionen #######
def analyze_tokens(data_by_lang, top_n=30):
    data = {}

    for lang in ["de", "en"]:
        sentences = data_by_lang[lang]["sentences"]
        tokens = [word for sent in sentences for word in sent]
        counts = Counter(tokens).most_common(top_n)

        data[(lang, "Token")] = [w for w, _ in counts]
        data[(lang, "Count")] = [c for _, c in counts]

    df = pd.DataFrame(data)
    df.columns = pd.MultiIndex.from_tuples(df.columns)

    return style_token_table(
        df,
        split_after_col=1,
        count_cols=[("de", "Count"), ("en", "Count")],
        caption=f"Top {top_n} Tokens (de vs en)"
    )

def compare_top_tokens(data_topic, data_sentiment, top_n=30):
    """
    Vergleich Topic vs Sentiment für DE und EN gleichzeitig
    """

    def get_counts(data, lang):
        tokens = [
            word
            for sent in data[lang]["sentences"]
            for word in sent
        ]
        return Counter(tokens).most_common(top_n)

    # Counts holen
    topic_de = get_counts(data_topic, "de")
    sent_de  = get_counts(data_sentiment, "de")

    topic_en = get_counts(data_topic, "en")
    sent_en  = get_counts(data_sentiment, "en")

    # DataFrame bauen
    df = pd.DataFrame({
        ("de", "Topic", "Token"): [w for w, _ in topic_de],
        ("de", "Topic", "Count"): [c for _, c in topic_de],
        ("de", "Sentiment", "Token"): [w for w, _ in sent_de],
        ("de", "Sentiment", "Count"): [c for _, c in sent_de],

        ("en", "Topic", "Token"): [w for w, _ in topic_en],
        ("en", "Topic", "Count"): [c for _, c in topic_en],
        ("en", "Sentiment", "Token"): [w for w, _ in sent_en],
        ("en", "Sentiment", "Count"): [c for _, c in sent_en],
    })

    df.columns = pd.MultiIndex.from_tuples(df.columns)

    return df
    
def print_lda_topics(model, feature_names, n_top_words=10):
    """
    Gibt die wichtigsten Wörter je Topic aus.
    """
    for topic_idx, topic in enumerate(model.components_):
        top_indices = topic.argsort()[:-n_top_words - 1:-1]
        top_words = [feature_names[i] for i in top_indices]

        print(f"\nTopic {topic_idx + 1}:")
        print(", ".join(top_words))

def topics_matrix(model, feature_names, n_top_words=10, prefix="Topic"):
    """
    Erstellt eine Topic-Tabelle für ein einzelnes Modell.
    Spalten: Topic1, Topic2, ...
    Zeilen: Top-Wörter
    """
    topics = {}

    for i, topic in enumerate(model.components_):
        top_idx = topic.argsort()[-n_top_words:][::-1]
        topics[f"{prefix}{i+1}"] = [feature_names[j] for j in top_idx]

    return pd.DataFrame(topics)


def compare_topic_models_multiindex(models, n_top_words=10):
    first_model = next(iter(models.values()))["model"]
    n_topics = first_model.components_.shape[0]

    data = {}

    for topic_idx in range(n_topics):
        for method_name, d in models.items():
            topic = d["model"].components_[topic_idx]
            features = d["features"]

            top_idx = topic.argsort()[-n_top_words:][::-1]
            words = [features[i] for i in top_idx]

            data[(f"Topic {topic_idx+1}", method_name)] = words

    return pd.DataFrame(data)

def build_topic_comparison_tables(doc_topics, methods, lang):
    
    def to_word_list(words):
        if isinstance(words, list):
            return words
        return [w.strip() for w in str(words).split(",")]

    rows = []

    for method_key, lang_results in doc_topics.items():
        df = lang_results[lang]
        title = methods[method_key]["title"]

        # Topic → Keywords (ein Eintrag pro Topic)
        topic_map = (
            df.groupby("dominant_topic")["topic_keywords"]
            .first()
            .sort_index()
            .apply(to_word_list)
        )

        max_len = max(len(words) for words in topic_map)

        for i in range(max_len):
            row = {"Method": title if i == 0 else ""}

            for topic_id, words in topic_map.items():
                row[f"Topic {topic_id + 1}"] = words[i] if i < len(words) else ""

            rows.append(row)

    return pd.DataFrame(rows)

In [5]:
def batch_lemmatizer(texts, nlp, stopw, desc, n_process):
    sentences = []

    reviews = nlp.pipe(texts, batch_size=200, n_process=n_process)

    for review in tqdm(reviews, total=len(texts), desc=desc):
        lemmas = [
            token.lemma_.lower()
            for token in review
            if token.is_alpha
            and len(token) > 2
            and not token.is_stop
            and token.lemma_.lower() not in stopw
        ]

        if lemmas:
            sentences.append(lemmas)

    return sentences
            
def clean_and_tokenize(df, stopw_de, stopw_en, n_process):
    stopw_de = set(stopw_de)
    stopw_en = set(stopw_en)

    nlp_de = spacy.load("de_core_news_sm", disable=["parser", "ner"])
    nlp_en = spacy.load("en_core_web_sm", disable=["parser", "ner"])

    df_de = df[df["language"] == "de"]["review"].fillna("").astype(str)
    df_en = df[df["language"] == "en"]["review"].fillna("").astype(str)

    result = {}

    for lang, texts, nlp, stopw, desc in [
        ("de", df_de, nlp_de, stopw_de, "Deutsch verarbeiten"),
        ("en", df_en, nlp_en, stopw_en, "Englisch verarbeiten"),
    ]:
        sentences = batch_lemmatizer(
            texts=texts,
            nlp=nlp,
            stopw=stopw,
            desc=desc,
            n_process=n_process
        )

        documents = [" ".join(sentence) for sentence in sentences]
        vocabulary = sorted(set(word for sent in sentences for word in sent))
        index = {word: i for i, word in enumerate(vocabulary)}

        result[lang] = {
            "sentences": sentences,
            "documents": documents,
            "vocabulary": vocabulary,
            "index": index
        }

    return result

def vectorize(Vectorizer, data_by_lang):
    params = {
        "ngram_range": (1, 2),
        "min_df": 3,
        "max_df": 0.8,
        "dtype": np.float32,
    }

    results = {}

    for lang, data in data_by_lang.items():
        documents = data["documents"]

        vectorizer = Vectorizer(**params)
        matrix = vectorizer.fit_transform(documents)

        results[lang] = {
            "matrix": matrix,
            "vectorizer": vectorizer,
            "feature_names": vectorizer.get_feature_names_out(),
            "documents": documents,
            "sentences": data["sentences"],
            "vocabulary": data["vocabulary"],
            "index": data["index"]
        }

        print(f"{lang}: {matrix.shape[0]} Dokumente, {matrix.shape[1]} Features")

    return results

def assign_topics_with_keywords(model_result, method, n_top_words=10):
    """
    Einheitliche Topic-Zuordnung für LDA und LSA inkl. Top-Keywords.

    Erwartet in model_result:
        - "doc_topic_matrix"
        - "documents"
        - "model"
        - "features"

    Parameter:
        method (str): 'lda' oder 'lsa'
        n_top_words (int): Anzahl Top-Wörter pro Topic

    Rückgabe:
        DataFrame mit:
        - document
        - dominant_topic
        - topic_strength
        - topic_keywords
    """

    doc_topic_matrix = model_result["doc_topic_matrix"]

    # --- Topic-Zuordnung ---
    if method == "lda":
        dominant_topics = doc_topic_matrix.argmax(axis=1)
        topic_strengths = doc_topic_matrix.max(axis=1)

    elif method == "lsa":
        dominant_topics = np.abs(doc_topic_matrix).argmax(axis=1)
        topic_strengths = np.abs(doc_topic_matrix).max(axis=1)

    else:
        raise ValueError("method muss 'lda' oder 'lsa' sein")

    # --- Keywords pro Topic ---
    model = model_result["model"]
    features = model_result["features"]

    topic_keywords = {
        i + 1: ", ".join(
            features[j]
            for j in topic.argsort()[-n_top_words:][::-1]
        )
        for i, topic in enumerate(model.components_)
    }

    # --- DataFrame ---
    df = pd.DataFrame({
        "document": model_result["documents"],
        "dominant_topic": dominant_topics + 1,
        "topic_strength": topic_strengths
    })

    # Keywords mappen
    df["topic_keywords"] = df["dominant_topic"].map(topic_keywords)

    return df

def build_sentiment_df(model_result, lang, pos=0.05, neg=-0.05):
    """
    Berechnet Sentiment Score + Label pro Dokument.

    Parameter:
        model_result: enthält 'documents'
        lang: 'de' oder 'en'
        pos/neg: Schwellenwerte für Klassifikation

    Rückgabe:
        DataFrame mit:
        - document
        - sentiment_score
        - sentiment_label
    """

    analyzer_en = SentimentIntensityAnalyzer()
    analyzer_de = GerSentimentIntensityAnalyzer()
    
    documents = model_result["documents"]

    # richtigen Analyzer wählen
    if lang == "en":
        analyzer = analyzer_en
    elif lang == "de":
        analyzer = analyzer_de
    else:
        raise ValueError("Unsupported language")

    # Scores berechnen
    scores = [
        analyzer.polarity_scores(doc)["compound"]
        for doc in documents
    ]

    # DataFrame erstellen
    df = pd.DataFrame({
        "document": documents,
        "sentiment_score": scores
    })

    # Label direkt inline
    df["sentiment_label"] = df["sentiment_score"].apply(
        lambda s: "positive" if s >= pos else "negative" if s <= neg else "neutral"
    )

    return df


In [6]:
##########
# Variable Definition
##########
local_dir = "data"
path_reviews = f"{local_dir}/combined_reviews.csv"

# Manuelle Stopwörter (werden iterativ ergänzt) 
stopw_topic_manual_de, stopw_sentiment_manual_de = [],[]
stopw_topic_manual_en, stopw_sentiment_manual_en = [],[]

# Stoppwörter aus NTLK
stopw_de = stopwords.words("german")
stopw_en = stopwords.words("english") 

# Zusammengesetzte Stopwörter aus manuell und NTLK
stopw_topic_de = stopw_de + stopw_topic_manual_de
stopw_topic_en = stopw_en + stopw_topic_manual_en
stopw_sentiment_de = stopw_de + stopw_sentiment_manual_de
stopw_sentiment_en = stopw_en + stopw_sentiment_manual_en

############
# Import data to pandas dataframe
############
df_reviews = pd.read_csv(path_reviews)

df_reviews["language"] = df_reviews["origin"].map({
    "FragdenStaat": "de",
    "YELP": "en",
})

############
# Prepare Data
############

# --- Stopwort-Definitionen ---
stopword_updates = {
    "iter_1": {
        "de_common": [
            "august","information","art","aktuell","zahl","genannt","frage","antwort","fall",
            "stelle","rahmen","bitten","insbesondere","erachten","tatsächlich",
            "bayerisch","bayern","baydsg","bayuig","vig", # Verwaltungsbegriffe
            "lebensmittelbetriebe","routinekontrolle","rüb","avv","lfgb" # Standardanfragen
        ],
        "en_common": [
            "want","look","know","think","people","day","find","tell","ask","take","work"
        ],
        "en_topic_only": [
            "nice","well","delicious","friendly","definitely"
        ]
    },

    "iter_2": {
        "de_common": [
            "bitte","einschließlich","datum","folgend","liegen","antrag","behörde","anfrage",
            "projekt","dokument","erfolgen","zuständig","zuständigkeitsbereich","falls",
            "auskunft","entsprechend","befinden","angabe","betreffen",
            "elektronisch","öffentlich","soweit","überprüfen","registriert", # Juristisch
            "aktenauskunft","gesetz","umweltinformationsgesetz" # Juristisch
        ],
        "en_common": [
            "place","food","get","try","time","come","go",
            "need","way","say","staff","experience"
        ],
        "en_topic_only": [
            "good","great","like","love","little"
        ]
    },

    "iter_3": { 
        "de_common": [ # Hauptsächlich Verwaltungsfloskeln 
            "letzter","form","begründung","interesse",
            "sämtlicher","anzahl","gemeinde","handeln",
            "geplant","sinn","monat", "freundlich",
            "senden","mitteilen","grüße","häufig"
            "stellen","höhe","angeben"
        ],
        "en_common": [ 
            "lot","feel","thing","visit", "pretty",
            "long","area","minute","new"
        ],
        "en_topic_only": ["bad","amazing"]
    },
    
    "iter_4": { 
        "de_common": [ 
            "unverzüglich", "ausdrücklich", "häufig", 
            "vorab", "bewerten", "übersicht", "jährlich", "mühe",
            "danken", "verweisen", "zugänglich"
        ],
        "en_common": [ 
            "sure", "right", "review", "hour","leave"
        ],
        "en_topic_only": ["recommend", "enjoy"]
    },
    
    "iter_5": { # Nach BoW + LDA Topic Modeling
        "de_common": [ 
            "satz", "empfangsbestätigung", "widersprechen", "weiterzuleiten",
            "unterrichten", "weitergabe", "aufwand", "gebührenpflichtig",
            "herr", "geehrt", "dame", "gemäß", "beantragen","gmbh",
            "vorhanden", "vorliegen", "zugang", "angefragt",
            "gewähren", "fragdenstaat", "verfahren", "gesetzlich",
            "grund", "bezug", "mitteilung"
        ],
        "en_common": [],
        "en_topic_only": []
    },

    "iter_6": { # Nach 2tem BoW + LDA Topic Modeling Durchlauf
        "de_common": [ 
            "einfach", "somit", "spätestens", "stellen",
            "zusätzlich", "sofern", "aufgrund"
        ],
        "en_common": [],
        "en_topic_only": []
    }
}

for iteration in stopword_updates.values():

    # Deutsch (immer beide)
    stopw_topic_de += iteration["de_common"]
    stopw_sentiment_de += iteration["de_common"]

    # Englisch (gemeinsam)
    stopw_topic_en += iteration["en_common"]
    stopw_sentiment_en += iteration["en_common"]

    # Englisch (nur Topic)
    stopw_topic_en += iteration["en_topic_only"]

# ggf. Duplikate entfernen
stopw_topic_de = sorted(set(stopw_topic_de))
stopw_sentiment_de = sorted(set(stopw_sentiment_de))
stopw_topic_en = sorted(set(stopw_topic_en))
stopw_sentiment_en = sorted(set(stopw_sentiment_en))


# Daten für Topic Modeling
print("Cleaning Data - Topic Modeling")
data_topic = clean_and_tokenize(
    df_reviews,
    stopw_de=stopw_topic_de,
    stopw_en=stopw_topic_en,
    n_process=2 # ggf. Anpassen für schnellere Verarbeitung
)

# Daten für Sentimentanalyse
print("\nCleaning Data - Sentiment Analysis")
data_sentiment = clean_and_tokenize(
    df_reviews,
    stopw_de=stopw_sentiment_de,
    stopw_en=stopw_sentiment_en,
    n_process=2
)




Cleaning Data - Topic Modeling


Englisch verarbeiten: 100%|████████████████████████████████████████████████████████| 1000/1000 [00:12<00:00, 77.12it/s]



Cleaning Data - Sentiment Analysis


Englisch verarbeiten: 100%|████████████████████████████████████████████████████████| 1000/1000 [00:13<00:00, 75.44it/s]


In [19]:
top_n = 15
compare_top_tokens_table = compare_top_tokens(data_topic, data_sentiment, top_n)
display(style_compare_top_tokens(compare_top_tokens_table, caption=f"Top {top_n} Tokens: Topic vs Sentiment (de/en)"))

In [20]:
############
# Create Vectors with BoW & TF-IDF
# Separate Pipelines: Topic Modeling vs. Sentiment
############

print("Creating Vectors - Topic Modeling")
vectors_topic = {}
vectors_topic["bow_by_lang"] = vectorize(CountVectorizer, data_topic)
vectors_topic["tfidf_by_lang"] = vectorize(TfidfVectorizer, data_topic)

print("\nCreating Vectors - Sentiment Analysis")
vectors_sentiment = {}
vectors_sentiment["bow_by_lang"] = vectorize(CountVectorizer, data_sentiment)
vectors_sentiment["tfidf_by_lang"] = vectorize(TfidfVectorizer, data_sentiment)

Creating Vectors - Topic Modeling
de: 1000 Dokumente, 3774 Features
en: 1000 Dokumente, 2416 Features
de: 1000 Dokumente, 3774 Features
en: 1000 Dokumente, 2416 Features

Creating Vectors - Sentiment Analysis
de: 1000 Dokumente, 3774 Features
en: 1000 Dokumente, 2641 Features
de: 1000 Dokumente, 3774 Features
en: 1000 Dokumente, 2641 Features


In [57]:
############
# Topic Modeling
############

# Zentrale Parameter für alle Topic-Modelle
n_topics = 10          # Anzahl der Topics
n_top_words = 15       # Anzahl der Top-Wörter pro Topic
max_iter = 20          # Anzahl der Trainingsdurchläufe für LDA
batch_size = 500       # Batch-Größe für das Online-Learning bei LDA

# Zentrale Konfiguration aller Topic-Modelle
methods = {
    "bow_lda": {
        "title": "LDA BoW",
        "vectors_by_lang": vectors_topic["bow_by_lang"],
        "method_type": "lda",
    },
    "tfidf_lda": {
        "title": "LDA TF-IDF",
        "vectors_by_lang": vectors_topic["tfidf_by_lang"],
        "method_type": "lda",
    },
    "tfidf_lsa": {
        "title": "LSA TF-IDF",
        "vectors_by_lang": vectors_topic["tfidf_by_lang"],
        "method_type": "lsa",
    },
}

# Jedes konfigurierte Modell trainieren
for method_key, config in methods.items():

    # Hier werden die trainierten Ergebnisse je Sprache gespeichert
    config["topics_by_lang"] = {}

    # Sprachgetrenntes Training, für "de" und "en"
    for lang, data in config["vectors_by_lang"].items():
        X = data["matrix"]  # Dokument-Term-Matrix der jeweiligen Sprache

        # LDA-Modell trainieren
        if config["method_type"] == "lda":
            model = LatentDirichletAllocation(
                n_components=n_topics,
                random_state=42,
                learning_method="online",
                max_iter=max_iter,
                evaluate_every=-1
            )

            # Manuelles Online-Training über mehrere Iterationen und Batches
            for _ in tqdm(range(max_iter), desc=f'{config["title"]} Topic tranieren ({lang})'):
                for i in range(0, X.shape[0], batch_size):
                    model.partial_fit(X[i:i + batch_size])

            # Dokumente in Topic-Wahrscheinlichkeiten transformieren
            topic_matrix = model.transform(X)

        # LSA-Modell trainieren
        elif config["method_type"] == "lsa":
            model = TruncatedSVD(
                n_components=n_topics,
                random_state=42
            )

            # Dokumente direkt in latente Topic-Komponenten transformieren
            topic_matrix = model.fit_transform(X)
            print(f'{config["title"]} Topic trainiert ({lang})')


        # Modell- und Ergebnisdaten zentral im methods-Dict speichern
        config["topics_by_lang"][lang] = {
            "model": model,
            "doc_topic_matrix": topic_matrix,
            "matrix": X,
            "features": data["feature_names"],
            "documents": data["documents"]
        }

LDA TF-IDF Topic tranieren (en): 100%|█████████████████████████████████████████████████| 20/20 [00:03<00:00,  6.38it/s]


LSA TF-IDF Topic trainiert (de)
LSA TF-IDF Topic trainiert (en)


In [69]:
def get_structure(d, max_depth=3, current_depth=0):
    if current_depth >= max_depth:
        return "..."

    if isinstance(d, dict):
        return {
            k: get_structure(v, max_depth, current_depth + 1)
            for k, v in d.items()
        }

    return type(d).__name__


pprint(get_structure(next(iter(methods.values())), max_depth=3))

{'method_type': 'str',
 'title': 'str',
 'topics_by_lang': {'de': {'doc_topic_matrix': '...',
                           'documents': '...',
                           'features': '...',
                           'matrix': '...',
                           'model': '...'},
                    'en': {'doc_topic_matrix': '...',
                           'documents': '...',
                           'features': '...',
                           'matrix': '...',
                           'model': '...'}},
 'vectors_by_lang': {'de': {'documents': '...',
                            'feature_names': '...',
                            'index': '...',
                            'matrix': '...',
                            'sentences': '...',
                            'vectorizer': '...',
                            'vocabulary': '...'},
                     'en': {'documents': '...',
                            'feature_names': '...',
                            'index': '...',
         

In [61]:
############
# Topic + Sentiment Analyse
############


# 1. Topic-Zuordnung je Modell
doc_topics = {
    method: {
        lang: assign_topics_with_keywords(
            result,
            method=config["method_type"]
        )
        for lang, result in config["topics_by_lang"].items()
    }
    for method, config in methods.items()
}

# 2. Sentiment je Sprache berechnen
# Sentiment basiert auf den Sentiment-Dokumenten, nicht auf BoW/TF-IDF
sentiments = {
    lang: build_sentiment_df(result, lang=lang)
    for lang, result in vectors_sentiment["tfidf_by_lang"].items()
}

# 3. Topic-Zuordnung + Sentiment verbinden
doc_topics_sentiment = {
    method: {
        lang: df_topics.merge(
            sentiments[lang],
            on="document",
            how="left"
        )
        for lang, df_topics in lang_results.items()
    }
    for method, lang_results in doc_topics.items()
}

# 4. Aggregierte Topic/Sentiment-Tabelle je Modell
topic_sentiment_summary = {
    method: {
        lang: (
            df.groupby("dominant_topic")
            .agg(
                documents=("document", "count"),
                avg_sentiment=("sentiment_score", "mean"),
                positive=("sentiment_label", lambda x: (x == "positive").sum()),
                neutral=("sentiment_label", lambda x: (x == "neutral").sum()),
                negative=("sentiment_label", lambda x: (x == "negative").sum()),
                topic_keywords=("topic_keywords", "first")
            )
            .reset_index()
        )
        for lang, df in lang_results.items()
    }
    for method, lang_results in doc_topics_sentiment.items()
}

In [66]:
# 5. Ausgabe: BoW/LDA Topic + Sentiment

for lang in ["de", "en"]:
    if lang == "de":
        pos_threshold, neg_threshold = 0.5, -0.5
    elif lang == "en":
        pos_threshold, neg_threshold = 0.1, -0.1

    for method_key, config in methods.items():
        display(style_topic_sentiment(
            topic_sentiment_summary[method_key][lang],
            f"{config["title"]} ({lang}) - Topic & Sentiment Matrix",
            pos_threshold=pos_threshold,
            neg_threshold=neg_threshold
        ))

dominant_topic,documents,avg_sentiment,positive,neutral,negative,topic_keywords
1,3790,0.509224,3775,5,10,"verfahrensstand, baubeginn, bebauungsplanverfahr, gemeindegebiet, gemarkung, aktueller, bebauungsplan, aktueller verfahrensstand, satzungsbeschluß, verfolgen"
2,330,0.629269,275,28,27,"unterlage, politisch, thema, ministerium, kommunikation, vertrag, berechtigt, geben, nutzen, aktivität"
3,376,0.161292,95,272,9,"verbraucherinformation, kosten, umweltinformation, gebühr, fallen, geringfügig, ablauf, erbeten, einschlägig, erbeten ablauf"
4,9026,-0.862404,0,1,9025,"betrieb, bußgeld, werbung, plattform, verstoß, unzulässig, erfolgt, berücksichtigung, herstellen, antworten"
5,44,0.332309,29,7,8,"prüfung, landesamt, person, bestehend, begründen, biber, landesamt verfassungsschutz, bewertung, gebiet, erlaubnis"
6,121,0.284903,60,47,14,"behördlich, tier, bundesland, aufschlüsseln, dokumentiert, betrieb, haus, überwachung, schlachthof, maßnahme"
7,384,0.273303,324,27,33,"münchen, straße, stadt, stehen, person, finden, maßnahme, derzeit, landratsamt, laut"
8,127,0.737516,120,6,1,"arzt, medizinisch, meldung, verdacht, unerwünscht, unterweisung, bevölkerung, deutsch, hintergrund, verfügung"
9,164,0.766540,140,17,7,"waffenschein, afd, verfassungsfeindlich, waffenbesitzkarte, mitglied, waffenbesitzkarte waffenschein, verfassungsfeindlich vereinigung, person, vereinigung, notfallsanitäter"
10,30,0.768963,25,5,0,"fördermeng, genehmigt, unternehmen, genehmigt fördermeng, kommunal, wasser, genehmigung, menge, entrichten, kalenderjahr"


dominant_topic,documents,avg_sentiment,positive,neutral,negative,topic_keywords
1,679,0.295173,410,267,2,"aufgabe, erwartungshorizonte, fach, aufgabe erwartungshorizonte, erwartungshorizonte lösung, lösung, lösung fach, maßregelvollzug, waffenschein, amtsärztlich"
2,276,0.713903,243,24,9,"gutachten, klimaneutralität, aktivität, thema, lehrkraft, protokoll, ministerium, liste, grundwasser, dienst"
3,123,0.497493,100,10,13,"verbraucherinformation, kosten, einschlägig, kosten geringfügig, bürgeranfrage behandeln, geringfügig, behandeln, geringfügig gebühr, gebühr fallen, einschlägig bürgeranfrage"
4,9026,-0.862404,0,1,9025,"bußgeld, betrieb, werbung, social media, werbung social, social, herstellen lebensmittelkontrolleur, kontrollhäufigkeit, kontrollierend, vollzeitäquivalente fte"
5,61,0.589321,43,15,3,"fördermeng, genehmigt, genehmigt fördermeng, kommunal, unternehmen, menge, entrichten, kommunal trinkwasser, trinkwasser, kalenderjahr"
6,169,0.284769,107,19,43,"münchen, polizei, straße, stadt, einsatz, maßnahme, bereich, unterlage, gerne, patient"
7,83,0.356799,55,15,13,"jva, frau, wieviel, besuch, nürnberg, kontrolle, unterbringung, bayrischzell, umgang, hiermit"
8,147,0.540519,106,36,5,"behördlich überwachung, dokumentiert geschlachtet, berichtsjahr, schlachthof, tierart, geschlachtet tier, geschlachtet, abgeschlossen berichtsjahr, dokumentiert, tier"
9,3707,0.507387,3689,7,11,"verfahrensstand, bebauungsplanverfahr, baubeginn, gemeindegebiet, aktueller verfahrensstand, gemarkung, aktueller, bebauungsplan, verfolgen, satzungsbeschluß"
10,121,0.512999,90,21,10,"persönlich, feuerwehr, stadt, gymnasium, berechtigt, weisung, steuer, person, stand, bay"


dominant_topic,documents,avg_sentiment,positive,neutral,negative,topic_keywords
1,9041,-0.860765,5,8,9028,"bußgeld, betrieb, durchzuführend risikobeurteilung, durchzuführend, zuständigkeitsgebiet veröffentlichen, zuständigkeitsgebiet, kontrollhäufigkeit erfolgt, kontrollierend, kontrollhäufigkeit, werbeaussag unzulässig"
2,3395,0.523982,3377,6,12,"verfahrensstand, bebauungsplanverfahr, baubeginn, gemeindegebiet, aktueller verfahrensstand, satzungsbeschluß, offiziell baubeginn, bebauungsplanentwurf, aufstellungsbeschluss, bebauungsplanverfahr projektname"
3,525,0.515624,408,65,52,"verbraucherinformation, kosten, umweltinformation, kosten geringfügig, geringfügig, bürgeranfrage behandeln, bürgeranfrage, behandeln, einschlägig bürgeranfrage, einschlägig"
4,90,0.458298,80,8,2,"fach, erwartungshorizonte lösung, aufgabe erwartungshorizonte, erwartungshorizonte, aufgabe, lösung, lösung fach, fach deutsch, deutsch, physik"
5,79,0.147247,27,42,10,"schlachthof, tier, dokumentiert geschlachtet, behördlich überwachung, geschlachtet tier, geschlachtet, tierart, abgeschlossen berichtsjahr, berichtsjahr behördlich, berichtsjahr"
6,291,0.047350,24,261,6,"maßregelvollzug, amtsärztlich begehung, klinik maßregelvollzug, krankenhaus klinik, bericht gutachten, amtsärztlich, gesundheitsamt krankenhaus, begehung gesundheitsamt, gutachten amtsärztlich, durchführen beziehen"
7,84,0.526356,68,5,11,"fördermeng, genehmigt, genehmigt fördermeng, unternehmen, menge, kalenderjahr, kommunal trinkwasser, trinkwasser, entrichten, wasser"
8,206,0.167028,200,2,4,"gesamtabrechnunge auflistung, explizit hinweisen, vorliegend dateiformat, gesamtabrechnunge, hinweisen personenbezogen, wirkstoffgehalt, wirkstoffgehalt maßregelvollzug, menge wirkstoffgehalt, maßregelvollzug einkaufen, bestellen"
9,332,0.340019,328,4,0,"gemarkung flurstücksnr, termin planunterlag, aufstellungsbeschluss offenlegung, kurz bebauungsplanverfahr, fortführung verfolgt, offenlegung bebauungsplanentwurf, bebauungsplan link, satzungsbeschluß termin, flurstücksnr, aktiv kurz"
10,349,0.816844,326,14,9,"waffenschein, verfassungsfeindlich vereinigung, verfassungsfeindlich, vereinigung, afd, waffenbesitzkarte, waffenbesitzkarte waffenschein, mitglied, alternative, verfassungsschutz"


dominant_topic,documents,avg_sentiment,positive,neutral,negative,topic_keywords
1,89,0.170443,3,2,2,"pizza, hair, use, close, absolutely, beef, flavor, serve, cut, pork"
2,69,-0.036443,2,1,4,"nail, kitchen, open, owner, drive, service, later, onion, finally, fan"
3,78,0.442750,6,0,2,"hotel, room, check, treat, awesome, seating, wedding, happy, burger, guest"
4,197,0.154273,6,1,4,"fresh, order, salad, bread, meat, service, menu, eat, chicken, wine"
5,62,-0.155963,1,1,6,"clean, stay, large, door, type, room, restaurant, flavor, italian, star"
6,90,-0.025100,3,1,3,"order, wait, taco, burger, drink, atmosphere, bbq, eat, bean, tip"
7,85,0.818867,3,0,0,"lunch, chicken, drink, beer, bar, hot, sandwich, salad, tasty, fry"
8,84,0.341940,4,0,1,"price, store, service, sushi, month, reasonable, selection, roll, buy, sandwich"
9,138,-0.087545,5,0,6,"order, table, friend, night, bar, restaurant, sit, service, server, cake"
10,108,0.144400,9,1,6,"car, service, customer, coffee, wait, call, pay, bring, line, change"


dominant_topic,documents,avg_sentiment,positive,neutral,negative,topic_keywords
1,121,0.014700,4,0,4,"pizza, order, table, hair, drink, wait, service, restaurant, awesome, vegetarian"
2,68,0.072671,3,0,4,"wine, awesome, service, wedding, music, beer, beautiful, wonderful, tour, ring"
3,45,0.866950,2,0,0,"sushi, burger, donut, frozen, eat, kid, treat, pet, clean, chicken"
4,185,-0.086800,2,1,4,"salad, order, taco, service, bread, fish, sauce, meat, chicken, eat"
5,137,0.001271,5,3,9,"room, store, car, stay, hotel, clean, nail, helpful, nashville, service"
6,92,0.136330,5,1,4,"pizza, fresh, fry, best, burger, flavor, line, game, wall, location"
7,78,0.688000,6,0,0,"roll, sushi, tea, soup, chinese, oyster, lunch, order, pho, seafood"
8,82,0.351850,4,0,2,"chicken, ice, coffee, ice cream, cream, reasonable, cute, sandwich, service, awesome"
9,67,-0.106300,2,0,2,"cake, change, service, price, fair, server, relaxed, rude, customer service, indian"
10,125,0.082244,9,2,5,"wait, service, bar, menu, car, call, burger, drink, oil, knowledgeable"


dominant_topic,documents,avg_sentiment,positive,neutral,negative,topic_keywords
1,615,0.082061,19,2,17,"order, service, wait, restaurant, burger, eat, drink, price, chicken, pizza"
2,76,0.120590,5,1,4,"burger, pizza, chicken, salad, fry, order, sauce, cheese, beer, fresh"
3,39,0.002517,2,1,3,"pizza, beer, car, crust, order pizza, thin, door, plain, stay, delivery"
4,28,0.690800,1,0,0,"sushi, service, roll, price, pizza, chicken, fresh, customer service, eat, favorite"
5,40,0.109425,2,0,2,"burger, service, car, sushi, fry, customer, customer service, order, excellent, slow"
6,33,0.509267,2,0,1,"order, drink, bar, table, restaurant, night, beer, sit, server, sushi"
7,12,nan,0,0,0,"sushi, room, roll, burger, hotel, pizza, clean, stay, beer, breakfast"
8,46,-0.075250,1,1,2,"beer, service, wine, bar, awesome, selection, price, drink, store, atmosphere"
9,37,0.261440,3,1,1,"room, hotel, excellent, service, chicken, stay, sauce, dinner, dish, awesome"
10,74,0.188792,7,1,4,"breakfast, service, coffee, awesome, sandwich, excellent, wait, pancake, breakfast sandwich, line"


In [68]:
# 6. Ausgabe: Topics per Method/ Language
topic_table_de = build_topic_comparison_tables(doc_topics,methods,"de")
topic_table_en = build_topic_comparison_tables(doc_topics,methods,"en")

display(style_topic_comparison(topic_table_de, "Topic Vergleich - DE"))
print()
display(style_topic_comparison(topic_table_en, "Topic Vergleich - EN"))

Method,Topic 2,Topic 3,Topic 4,Topic 5,Topic 6,Topic 7,Topic 8,Topic 9,Topic 10,Topic 11
LDA BoW,verfahrensstand,unterlage,verbraucherinformation,betrieb,prüfung,behördlich,münchen,arzt,waffenschein,fördermeng
,baubeginn,politisch,kosten,bußgeld,landesamt,tier,straße,medizinisch,afd,genehmigt
,bebauungsplanverfahr,thema,umweltinformation,werbung,person,bundesland,stadt,meldung,verfassungsfeindlich,unternehmen
,gemeindegebiet,ministerium,gebühr,plattform,bestehend,aufschlüsseln,stehen,verdacht,waffenbesitzkarte,genehmigt fördermeng
,gemarkung,kommunikation,fallen,verstoß,begründen,dokumentiert,person,unerwünscht,mitglied,kommunal
,aktueller,vertrag,geringfügig,unzulässig,biber,betrieb,finden,unterweisung,waffenbesitzkarte waffenschein,wasser
,bebauungsplan,berechtigt,ablauf,erfolgt,landesamt verfassungsschutz,haus,maßnahme,bevölkerung,verfassungsfeindlich vereinigung,genehmigung
,aktueller verfahrensstand,geben,erbeten,berücksichtigung,bewertung,überwachung,derzeit,deutsch,person,menge
,satzungsbeschluß,nutzen,einschlägig,herstellen,gebiet,schlachthof,landratsamt,hintergrund,vereinigung,entrichten
,verfolgen,aktivität,erbeten ablauf,antworten,erlaubnis,maßnahme,laut,verfügung,notfallsanitäter,kalenderjahr


Method,Topic 2,Topic 3,Topic 4,Topic 5,Topic 6,Topic 7,Topic 8,Topic 9,Topic 10,Topic 11
LDA BoW,pizza,nail,hotel,fresh,clean,order,lunch,price,order,car
,hair,kitchen,room,order,stay,wait,chicken,store,table,service
,use,open,check,salad,large,taco,drink,service,friend,customer
,close,owner,treat,bread,door,burger,beer,sushi,night,coffee
,absolutely,drive,awesome,meat,type,drink,bar,month,bar,wait
,beef,service,seating,service,room,atmosphere,hot,reasonable,restaurant,call
,flavor,later,wedding,menu,restaurant,bbq,sandwich,selection,sit,pay
,serve,onion,happy,eat,flavor,eat,salad,roll,service,bring
,cut,finally,burger,chicken,italian,bean,tasty,buy,server,line
,pork,fan,guest,wine,star,tip,fry,sandwich,cake,change
